# Run 2.1 — continuing the ladder from $\beta = 0.5422$

Continuation of [`flux_dev_strong_tilt_sweep_2.ipynb`](flux_dev_strong_tilt_sweep_2.ipynb) /
[`run_695271.ipynb`](run_695271.ipynb), which reached $\beta = 0.5422$ of 1.0
($m_{\text{eff}} = 1.63$ of 3) in 54 sweeps before its 8 h deadline. This notebook picks up that
particle cloud rather than restarting from $p$, saving the 5 levels already paid for.

> **First attempt (job 697271) aborted on the reward-reproducibility guard.** Slurm scheduled it on an
> L40S while 695271 had run on an A6000, and bf16 FLUX does not reproduce across GPU architectures.
> See *What must NOT change* below for the diagnosis and the two changes that close it.

## How the resume works

The tempering path is $\pi_\beta \propto p \cdot e^{\beta \lambda f}$. Substituting
$\beta = \beta_0 + (1-\beta_0)u$ for $u \in [0,1]$:

$$\pi \;\propto\; p\,e^{\beta_0 \lambda f} \cdot e^{(1-\beta_0)\lambda u f} \;=\; \pi_{\beta_0}\cdot e^{\lambda' u f},
\qquad \lambda' = (1-\beta_0)\lambda$$

So the residual leg is an ordinary SMC run from $u=0$ to $u=1$ whose **starting distribution is the saved
cloud** and whose tilt is $\lambda' = (1 - 0.5422) \times 276.59 = 126.61$. Nothing in
`adaptive_tempering_smc.py` needs changing — the SMC only ever uses $\Delta\beta \cdot \lambda \cdot f$, and
that product is preserved. Only `PCNKernel.init` is overridden, to return the saved state instead of
sampling fresh.

**Consequence: `lam_eff` and `beta` reported by the SMC are relative to the residual leg** and must be
remapped before they mean anything:

$$\lambda_{\text{eff}}^{\text{true}} = \beta_0\lambda + \lambda_{\text{eff}}^{\text{run}} = 149.97 + \lambda_{\text{eff}}^{\text{run}},
\qquad \beta^{\text{true}} = \beta_0 + u(1-\beta_0)$$

The results and image cells below do this; without it every row reads low by 150 (1.63σ).

## What must NOT change

The resume is only valid against a **reproducible reward**. `R = 64`, `N_GAMMA = 30`, `NUM_EPS = 3`, the
reference seed (`manual_seed(7)`), the distance seed (`123`) and `MAX_DENOISER_ROWS = 24` are all carried
over unchanged. As a check, `f(X)` is **recomputed** at init and compared against the `fX` stored in the
checkpoint; a material disagreement aborts the run rather than sampling from a subtly different target.

**The GPU is part of the reward.** This is what job 697271 discovered the hard way. Every knob above
matched, yet `f` came back different:

| | 695271 (source) | 697271 (first resume) |
|---|---|---|
| card | NVIDIA RTX A6000 (Ampere) | NVIDIA L40S (Ada) |
| ref bank build | 1404 s | 994 s |
| `max\|\Delta f\|` | — | 1.76e-3 |

The transformer runs bf16, whose 8-bit mantissa carries only ~0.2% relative precision per operation, and
the two architectures select different tensor-core kernels and reduction orders. The drift is therefore
pure round-off — 0.17% of $f \approx 1.03$ — but it propagates through $G$ on the references, the reference
score bank and the normalizer, and because this reward is unusually concentrated ($\operatorname{std}_p(f)$
is only 1.5% of the mean) that round-off reads as 11% of a standard deviation. At $\lambda' = 126.6$ it is
worth up to 0.22 nats of log-weight, so the guard was right to refuse. Two changes close it:

1. **`--constraint=a6000`** in `flux_strong_tilt_2_1.slurm` (was `l40s|a6000`), so the score field is
   evaluated on the source run's architecture. Asserted in the setup cell against
   `src["config"]["gpu"]`, which fails ~2 min in rather than after the ~20 min bank build. Cost: 2 A6000
   nodes versus 6 L40S, so a longer queue wait.
2. **The reference latents are persisted** to `refs_R64_seed7.pt` and reloaded on every later leg, taking
   $G$ out of the reproducibility chain entirely.

The guard's tolerance was also wrong, independently of the hardware: `f_tol = 1e-4` *of*
$\operatorname{std}_p(f)$ demanded agreement to 1.5e-6 relative, three orders of magnitude below what bf16
can deliver on any card. It is now stated in **nats of log-weight** (`tol_nats = 0.05`) and applied to the
*spread* of the drift rather than its magnitude — $q \propto p\,e^{\lambda f}$ is invariant under
$f \to f + c$, so only the spread moves relative weights. A separate 1%-of-$f$ tripwire on the common
offset still catches a genuinely wrong knob.

## Changes for this leg

| | run 695271 | here |
|---|---|---|
| starting point | $p$ ($\beta = 0$) | **saved cloud at $\beta_0 = 0.5422$** |
| `lam` passed to SMC | 276.59 | **126.61** (residual) |
| `MAX_SWEEPS` | 24 | **40** |
| `--time` | 480 min | **360 min** |
| `--constraint` | `l40s\|a6000` | **`a6000`** (see above) |
| references | generated by $G$ | **loaded from `refs_R64_seed7.pt`** |
| probe for $\lambda_s$ | 32 draws | **skipped** — $\lambda_s = 92.195$ read from the checkpoint |

`MAX_SWEEPS` 24 → 40 is precautionary, not a fix for an observed problem: the cap was **never reached**
(12/11/20/11 sweeps), so it was not binding. But acceptance fell 0.75 → 0.62 → 0.38 → 0.12 across the four
levels, so it is likely to start binding as $\lambda$ rises, and a silently truncated rejuvenation at high
tilt is the failure mode worth pre-empting.

The probe is skipped because $\lambda_s$ is a property of the (unchanged) reward and $p$, already measured
on 32 draws; re-measuring would cost ~14 min and introduce a second estimate for no gain.

## Outputs are tagged

Every output path carries `RUN_TAG = "2_1"`. The un-tagged names are derived from `N`/`M_TILT`/`SEED`,
which are unchanged on a resume — so without the tag this notebook would overwrite
`partial_N16_m3_seed101.pt` **on its first sweep**, i.e. destroy the very checkpoint it is resuming from.
An assertion below enforces that the source and destination paths differ.

## Expected reach — $\beta = 1$ is unlikely in 6 h

Remaining $\beta$ is 0.4578. After setup (~0.5 h) and reserve (1 h), 6 h buys ~4.5 h of sweeping ≈ 40 sweeps:

| | levels | $\beta$ reached |
|---|---|---|
| optimistic ($\Delta\beta = 0.136$, 13.5 sweeps/level, as 695271) | 2.9 | **0.94** |
| likely ($\Delta\beta$ −30%, 20 sweeps/level as acceptance falls) | 2.0 | **0.73** |
| pessimistic ($\Delta\beta$ halves, sweeps → cap 40) | 1.0 | 0.61 |

So this leg **advances** the ladder rather than completing it. That is a deliberate choice of a shorter
queue slot over a single long one: a further resume from this run's own checkpoint is the intended
continuation, and it will be cheaper than this one because `s` (the adapted pCN step size) is now persisted
and will carry over instead of restarting from $s_0$.


In [ ]:
import faulthandler
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

# --- hang watchdog --------------------------------------------------------------------------------
# Job 754633 held an A6000 for 6 h and produced no evidence of where it stalled. This dumps the Python
# stack of every thread to stderr (the job's .err) without killing the process, so a hang names its
# own frame. It is what identified the 754633 stall as `pipe.to(DEVICE)` paging memory-mapped
# safetensors over NFS.
#
# 30 min, not 10: at a 10 min cadence a 7 h run emits ~40 tracebacks and buries the SMC's own
# diagnostics in the .err. Any hang worth diagnosing outlives one interval by a wide margin -- the
# 755512 probe was already unambiguous after two dumps -- so the wider cadence localizes just as well.
faulthandler.enable()
faulthandler.dump_traceback_later(1800, repeat=True, exit=False)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
SWEEP_DIR = os.path.join(REPO, "notebooks", "flux_lambda_sweep_strong_2")
os.makedirs(SWEEP_DIR, exist_ok=True)

# --- output tagging -------------------------------------------------------------------------------
# EVERY output path carries RUN_TAG. The untagged names are derived from N/M_TILT/SEED, none of which
# change on a resume, so an untagged run would rewrite partial_N16_m3_seed101.pt -- its own resume
# source -- on the first sweep, and a later crash would leave nothing to fall back to.
RUN_TAG = "2_1"
SOURCE_CKPT = os.path.join(SWEEP_DIR, "partial_N16_m3_seed101.pt")   # read-only here
DECODED_DIR = os.path.join(SWEEP_DIR, f"decoded_{RUN_TAG}")
GRID_PNG = os.path.join(SWEEP_DIR, f"strong_tilt_grid_{RUN_TAG}.png")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call, so it cannot be interrupted from outside; the kernel checks
# DEADLINE after every sweep instead. RESERVE_H is held back for decoding and rendering.
# Under Slurm, SLURM_JOB_END_TIME is the job's real end, so the deadline tracks --time automatically.
#
# 7 h (was 6). The extra hour was added to absorb the node-local staging pass, but staging is now a
# persistent per-node cache (see the slurm script): job 755539 paid 257 min for it ONCE, and a cache
# hit costs nothing. So on a warm node the budget is ~30 min of setup (model load + reference bank +
# the f-recompute) plus the 1 h reserve, leaving ~5.5 h of sweeping, i.e. ~48 sweeps. Remaining beta
# is 0.4578, so at 695271's measured d_beta = 0.136 and 13.5 sweeps/level that is ~3.5 levels ->
# beta ~0.95; with acceptance falling (d_beta -30%, 20 sweeps/level) ~2.4 levels -> beta ~0.77.
#
# BETA = 1 REMAINS UNLIKELY but is no longer out of reach on the optimistic branch. A further resume
# from this run's OWN checkpoint is still the intended continuation, and `s` is persisted so the
# adapted pCN step size carries over instead of restarting from s0.
WALL_BUDGET_H = 7.0            # keep in sync with --time in flux_strong_tilt_2_1.slurm (420 min)
RESERVE_H = 1.0                # held back for VAE decoding, rendering, and writing the notebook
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

from huggingface_hub import auth_check

auth_check("black-forest-labs/FLUX.1-dev")
print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"SMC deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"run tag {RUN_TAG!r}   resuming from {os.path.basename(SOURCE_CKPT)}")
print("FLUX.1-dev access: OK")


In [ ]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"          # canonical id, kept for the CONFIG record
PROMPT = "A dog"

GUIDANCE = 1.5

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536  (1024^2 would be 262144, and pCN mixes worse)

# Job 754633 hung for 6 h inside `pipe.to(DEVICE)`. HF_HOME lives on an NFSv3 mount (rsize=32768,
# measured 28.9 MB/s sequential) and safetensors are MEMORY-MAPPED, so `.to(cuda)` is where the 24 GB
# of weights actually get read -- as random 32 KB NFS reads, which is 10-50x worse than sequential.
# The slurm script now stages the snapshot to node-local disk and exports FLUX_LOCAL_PATH, making
# this a local-disk read at GB/s. Falling back to MODEL_ID keeps the notebook runnable standalone
# (slowly) outside the job script.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
print(f"loading from {'STAGED ' + MODEL_PATH if MODEL_PATH != MODEL_ID else 'the HF cache (NFS)'}",
      flush=True)

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded; guidance={GUIDANCE} (baseline used 3.5)  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)


In [ ]:
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.base import edm_generator

SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0
N_STEPS = 8               # Heun ODE steps inside G (15 denoiser evals: 2/step, minus the last corrector)

# --- denoiser batch cap ---------------------------------------------------------------------------
# MEASURED, job 695266 on this exact card/model/resolution (see probe_denoiser_batch.py):
#
#     rows      24     48     96    128    192
#     empty     ok     ok     ok     ok    OOM
#     +bank     ok     ok     ok     ok    OOM      <- bank = the 1.36 GB resident ref score bank
#     free GB 15.8    8.2    2.3    0.6      -      (with the bank held)
#     s/row  0.259  0.251  0.255  0.263      -
#
# R=64 with NUM_EPS=3 needs 192 rows for the reference-bank build (score_bank -> _scores_at_gamma
# issues ONE call of NUM_EPS*R rows per gamma interval), which is exactly the width that OOMs -- and
# the width that killed job 694256. 694278 never exceeded 24 rows because pre-migration R was a loop
# counter, not a batch dimension.
#
# 24 rather than 128: s/row is FLAT across the range, so a narrow cap costs nothing in throughput
# (same number of forward passes, same cost each) and 128 would leave 0.6 GB free for an 8 h run.
# Fixed for the whole run, so f stays a deterministic function of x (CLAUDE.md invariant 1).
MAX_DENOISER_ROWS = 24

# Both must be 2-D in diffusers 0.39; _prepare_latent_image_ids takes the *packed* (half) grid.
img_ids = FluxPipeline._prepare_latent_image_ids(1, H // 2, W // 2, DEVICE, DTYPE)
txt_ids = text_ids if text_ids.ndim == 2 else text_ids[0]


def denoiser(x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """EDM denoiser D(x_sigma, sigma) = E[X | x_sigma] backed by the FLUX transformer.

    Flow matching interpolates, EDM adds:  x_t = (1-t)x_0 + t*eps  vs  x_sigma = x_0 + sigma*eps.
    Factoring out (1-t) makes them identical under  t = sigma/(1+sigma),  x_t = x_sigma/(1+sigma).
    FLUX predicts v = eps - x_0, so  x_t - t*v = x_0  exactly -- no approximation.
    Validated by the baseline run: its lambda_eff=0 row decoded to clean, on-prompt images.

    At sigma=80, t = 80/81 ~ 0.988 and x_t ~ z -- unit-scale noise, which is what FLUX expects at
    t -> 1, so sigma_max=80 is the natural choice for these unit-variance latents.

    Returns fp32 (the transformer runs bf16): reward.x_refs pins dtype for the whole SMC, and bf16
    weight/ESS arithmetic would quietly corrupt the beta schedule.
    """
    b = x.shape[0]
    sig = sigma.reshape(b, 1, 1, 1).to(torch.float32)
    t = (sigma / (1.0 + sigma)).to(torch.float32)
    x_t = x.to(torch.float32) / (1.0 + sig)
    packed = FluxPipeline._pack_latents(x_t.to(DTYPE), b, C, H, W)
    with torch.no_grad():
        v = transformer(
            hidden_states=packed,
            timestep=t.to(DTYPE),                                   # forward() scales by 1000 itself
            guidance=torch.full((b,), GUIDANCE, device=x.device, dtype=torch.float32),
            pooled_projections=pooled_prompt_embeds.expand(b, -1).to(DTYPE),
            encoder_hidden_states=prompt_embeds.expand(b, -1, -1).to(DTYPE),
            txt_ids=txt_ids,
            img_ids=img_ids,
            return_dict=False,
        )[0]
    v = FluxPipeline._unpack_latents(v, IMG, IMG, 8).to(torch.float32)
    return (x_t - t.reshape(b, 1, 1, 1) * v).to(torch.float32)


# Wrap ONCE and build both consumers from the wrapper: this covers the generator path
# (heun_prob_flow -> edm_ode_step) and every score path (marginal_score -> score_fn), including the
# reference score_bank build, which is the widest call in the run.
denoiser_capped = chunked_denoiser(denoiser, MAX_DENOISER_ROWS)

G = edm_generator(denoiser_capped, img_shape=(C, H, W), sigma_min=SIGMA_MIN,
                  sigma_max=SIGMA_MAX, n_steps=N_STEPS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
print(f"generator + score ready   sigma in [{SIGMA_MIN}, {SIGMA_MAX}]  n_steps={N_STEPS}")
print(f"denoiser batch capped at {MAX_DENOISER_ROWS} rows "
      f"(measured ceiling 128, OOM at 192; s/row flat so the cap is free)")


In [ ]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- knobs: IDENTICAL to run 695271, because the reward must reproduce bit-for-bit ------------------
# A resume is only valid if f is the same function it was. R, N_GAMMA, NUM_EPS, the reference seed (7),
# the distance seed (123) and MAX_DENOISER_ROWS (24) are therefore all carried over unchanged -- the
# last one because the references are produced by G, and a different chunking would perturb them at
# round-off. Verified below by recomputing f on the saved cloud.
#
# MAX_SWEEPS is the one exception, 24 -> 40. Precautionary, not a fix: the cap was NEVER reached in
# 695271 (12/11/20/11 sweeps, all stopping on `decorr`), so it was not binding. But acceptance fell
# 0.75 -> 0.62 -> 0.38 -> 0.12, so it is likely to start binding as lambda rises, and a silently
# truncated rejuvenation at high tilt is the failure mode worth pre-empting.
R, N_GAMMA, NUM_EPS = 64, 30, 3
N = 16
MAX_SWEEPS = 40
M_TILT = 3.0
ESS_TARGET = 0.80
SEED = 101

# --- load the source checkpoint -------------------------------------------------------------------
src = torch.load(SOURCE_CKPT, map_location="cpu", weights_only=False)
src_levels = src["levels"]
init_entry = src_levels[-1]                       # the deepest level reached: beta = 0.5422
BETA0 = float(init_entry["beta"])
LAM_FULL = float(src["config"]["lam_max"])        # 276.59, the lambda the FULL ladder targets
lam_s = float(src["config"]["lam_s"])             # 92.195 -- a property of the (unchanged) reward and p
LAM_RESUME = (1.0 - BETA0) * LAM_FULL             # the residual leg's tilt

assert src["config"]["R"] == R and src["config"]["n_gamma"] == N_GAMMA, "reward config drifted"
assert src["config"]["num_eps"] == NUM_EPS and src["config"]["seed"] == SEED
assert init_entry["X"].shape[0] == N, f"checkpoint holds {init_entry['X'].shape[0]} particles, N={N}"
assert "Z" in init_entry.get("aux", {}), "checkpoint has no pCN latents; cannot resume a pCN chain"

# --- hardware identity: the check job 697271 did not have ------------------------------------------
# 697271 aborted on the reward guard because Slurm handed it an L40S while 695271 had run on an A6000.
# FLUX runs bf16 (8-bit mantissa, ~0.2% per-op resolution) and Ampere vs Ada select different
# tensor-core kernels and reduction orders, so identical denoiser inputs give different outputs at
# round-off; f drifted 1.76e-3, which is only 0.17% of f but 11% of std_p(f). The slurm script now
# pins --constraint=a6000; this asserts it here, ~2 min in, rather than letting the run discover it
# after the ~20 min reference-bank build. The source checkpoint predates this config field, so
# 695271's card is read from its slurm log (logs/flux-tilt2-695271.out: "NVIDIA RTX A6000").
GPU_NAME = torch.cuda.get_device_name(0)
SOURCE_GPU = src["config"].get("gpu", "NVIDIA RTX A6000")
assert GPU_NAME == SOURCE_GPU, (
    f"resume scheduled on {GPU_NAME!r} but the source ran on {SOURCE_GPU!r}; bf16 FLUX does not "
    f"reproduce across architectures. Fix --constraint in flux_strong_tilt_2_1.slurm.")

print(f"source: {os.path.basename(SOURCE_CKPT)}   {len(src_levels)} levels, "
      f"{src['sweeps_done']} sweeps done")
print(f"  hardware: {GPU_NAME} -- matches the source run")
print(f"  resuming from level {len(src_levels) - 1}: beta0={BETA0:.4f}  "
      f"lam_eff={BETA0 * LAM_FULL:.2f}  m_eff={BETA0 * LAM_FULL / lam_s:.2f}")
print(f"  residual leg: lam' = (1 - {BETA0:.4f}) * {LAM_FULL:.2f} = {LAM_RESUME:.2f}")
print(f"  logw at resume: max|logw| = {float(init_entry['logw'].abs().max()):.3e} "
      f"(zeros => the cloud is equally weighted, so starting at logw=0 is correct)")

# --- forward batches ------------------------------------------------------------------------------
print(f"\nforward batches, capped at {MAX_DENOISER_ROWS} rows "
      f"(measured on an A6000: 128 ok, 192 OOM -- job 695266):")
for _lbl, _rows in (("G(z) on refs", R), ("G(z) per sweep", N),
                    ("score: ref bank", NUM_EPS * R), ("score: sweep reward", NUM_EPS * N)):
    print(f"  {_lbl:22s} {_rows:4d} rows -> {-(-_rows // MAX_DENOISER_ROWS):2d} call(s) "
          f"x <= {min(_rows, MAX_DENOISER_ROWS)}")

# --- references: load from disk if available, else generate AND SAVE -------------------------------
# WHY THIS EXISTS. Job 697271 aborted here: it was scheduled on an L40S while 695271 had run on an
# A6000, and regenerating the references through a 12B bf16 transformer on a different architecture
# selects different kernels. S matched to 3 dp but f drifted by 11% of std_p(f) -- enough to move
# weight ratios by ~1.25x at lam' = 126.6. Persisting the reference LATENTS removes G from the
# reproducibility chain entirely, so every future leg gets a bit-identical reference set regardless
# of where it lands. (The score_fn still runs on whatever GPU is allocated, so `--constraint=a6000`
# in the slurm script pins the other half; the two together are what make the guard below pass.)
REFS_PATH = os.path.join(SWEEP_DIR, f"refs_R{R}_seed7.pt")
if os.path.exists(REFS_PATH):
    _rf = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
    latent_refs = _rf["x_refs"].to(DEVICE)
    assert _rf.get("gpu", GPU_NAME) == GPU_NAME, (
        f"refs were generated on {_rf['gpu']!r} but this job is on {GPU_NAME!r}; the latents reload "
        f"exactly, but the score bank built from them would not match the source run")
    print(f"\nreferences LOADED from {os.path.basename(REFS_PATH)} -- G is out of the loop")
else:
    gen = torch.Generator(device="cpu").manual_seed(7)      # SAME seed as 695271
    with torch.no_grad():
        latent_refs = G(torch.randn(R, LATENT_DIM, generator=gen).to(DEVICE))
    torch.save({"x_refs": latent_refs.detach().cpu(), "R": R, "seed": 7,
                "latent_dim": LATENT_DIM, "gpu": GPU_NAME}, REFS_PATH)
    print(f"\nreferences GENERATED and saved to {os.path.basename(REFS_PATH)} "
          f"on {GPU_NAME} -- future legs will load these")

S = latent_refs.std().item()
GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)
D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=123, score_fn=score_fn)

t0 = time.time()
reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
t_norm = time.time() - t0
print(f"refs {tuple(latent_refs.shape)}  S={S:.4f} (695271: 1.0150)  "
      f"gamma=[{GAMMA_LO:.4g}, {GAMMA_HI:.3g}] x {N_GAMMA}   ref bank {t_norm:.0f}s "
      f"(695271 on A6000: 1404s)")

GEN_ROWS = 2 * N_STEPS - 1
REW_ROWS = (N_GAMMA - 1) * NUM_EPS
_bank_rows = R * GEN_ROWS + (N_GAMMA - 1) * NUM_EPS * R
print(f"  implied {t_norm / _bank_rows:.3f} s/row from {_bank_rows} bank rows "
      f"(A6000 measured 0.251)")

# --- cost projection ------------------------------------------------------------------------------
# S_ROW measured on an A6000 by job 695266 (0.25), independently derived from 694278's 165 s sweep
# (0.250), and validated by 695271's own run to 2.5% (actual 420.2 s vs projected 410.1 s).
S_ROW = 0.251
t_sweep = N * (GEN_ROWS + REW_ROWS) * S_ROW
budget_s = (WALL_BUDGET_H - RESERVE_H) * 3600 - (time.time() - T_START)
sweeps_afford = budget_s / t_sweep
rem_beta = 1.0 - BETA0
print(f"\nprojected {t_sweep / 60:.1f} min/sweep at {S_ROW} s/row   "
      f"budget left {budget_s / 3600:.2f} h -> ~{sweeps_afford:.0f} sweeps")
print(f"  reach at {sweeps_afford:.0f} sweeps (remaining beta {rem_beta:.4f}):")
for lbl, dbeta, swl in (("optimistic (695271's d_beta and 13.5 sweeps/level hold)", 0.1356, 13.5),
                        ("likely (d_beta -30%, 20 sweeps/level as acc falls)", 0.0949, 20.0),
                        ("pessimistic (d_beta halves, sweeps -> cap)", 0.0678, float(MAX_SWEEPS))):
    _lv = sweeps_afford / swl
    _beta = min(1.0, BETA0 + _lv * dbeta)
    print(f"    {lbl:52s} {_lv:4.1f} levels -> beta {_beta:.3f}"
          + ("  <- completes" if _beta >= 1.0 else ""))
print(f"  -> beta=1 is UNLIKELY in {WALL_BUDGET_H:.0f} h; this leg advances the ladder. Every level is")
print(f"     checkpointed, and `s` is now persisted so a further resume keeps the adapted step size.")


In [ ]:
from creativity_measure import LevelSnapshot, PCNKernel, adaptive_tempering_smc_sample
# _State is private, but resuming is something the library does not (yet) expose: the alternative is
# calling PCNKernel.init and discarding its work, which costs a wasted G(z) + f(X) on 16 particles
# (~1632 rows, ~7 min). Constructing the state directly is cheaper and states the intent plainly.
from creativity_measure.adaptive_tempering_smc import _State

CONFIG = {
    "model_id": MODEL_ID, "img": IMG, "C": C, "H": H, "W": W, "latent_dim": LATENT_DIM,
    "vae_scaling_factor": VAE_SF, "vae_shift_factor": VAE_SHIFT,
    "N": N, "m_tilt": M_TILT, "lam_max": LAM_FULL, "lam_resume": LAM_RESUME, "lam_s": lam_s,
    "beta0": BETA0, "resumed_from": os.path.basename(SOURCE_CKPT), "run_tag": RUN_TAG,
    "ess_target": ESS_TARGET, "n_mcmc": None, "max_n_mcmc": MAX_SWEEPS, "seed": SEED,
    "prompt": PROMPT, "guidance": GUIDANCE, "n_steps": N_STEPS,
    "sigma_min": SIGMA_MIN, "sigma_max": SIGMA_MAX,
    "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS, "gamma_lo": GAMMA_LO, "gamma_hi": GAMMA_HI,
    "max_denoiser_rows": MAX_DENOISER_ROWS, "s_row": S_ROW, "t_norm_s": t_norm,
    # Recorded so the NEXT leg can assert its card against this one instead of falling back to a
    # hard-coded name. 695271 predates this field, which is why cell 4 needs the literal default.
    "gpu": GPU_NAME,
}
PARTIAL_CKPT = os.path.join(SWEEP_DIR, f"partial_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}.pt")
assert os.path.abspath(PARTIAL_CKPT) != os.path.abspath(SOURCE_CKPT), \
    "destination equals the resume source -- the first sweep would destroy it"


class DeadlineReached(RuntimeError):
    """Raised by the kernel when the wall-clock deadline passes, to stop the SMC cleanly."""


class CheckpointingPCNKernel(PCNKernel):
    """Persists the cloud every sweep and stops the run at a wall-clock deadline.

    Only *reads* state after `super().step()` has run, so no randomness is consumed and no value
    changes: the run is bit-identical to one using a plain PCNKernel.

    Keying by `beta` is the trick -- beta changes only between levels, so each entry is overwritten
    within a level and ends up holding that level's LAST sweep, i.e. exactly the post-rejuvenation
    cloud the final snapshot records. `init` is overridden too, so the beta=0 anchor is captured.
    Writes are temp-file + os.replace, so a kill mid-write cannot corrupt the last good file.

    `ancestors` is arange here (resampling happens outside the kernel); the final save has the real ones.
    `state.ess` is the level's post-reweight ESS, set by the SMC loop before resampling -- reading it
    here is what lets a deadline-truncated run report ESS/N instead of NaN.
    `self.s` is the adapted pCN step size; without persisting it a resume restarts from s0 and has to
    re-adapt, discarding what the previous leg learned about the local geometry.
    """

    def __init__(self, *a, path, config, deadline, save_z=True, **kw):
        super().__init__(*a, **kw)
        self._path, self._config, self._deadline, self._save_z = path, config, deadline, save_z
        self._latest, self._sweeps_at, self._sweeps = {}, {}, 0

    def _record(self, state, *, beta, lam, acc):
        e = {"beta": beta, "lam_eff": beta * lam,
             "X": state.X.detach().to("cpu").clone(), "fX": state.fX.detach().to("cpu").clone(),
             "logw": state.logw.detach().to("cpu").clone(),
             "ess": float(state.ess),          # level ESS, pre-resample (nan at beta=0)
             "s": float(self.s),               # adapted pCN step size, for a further resume
             "ancestors": torch.arange(state.X.shape[0]),
             "sweeps_at_beta": self._sweeps_at.get(beta, 0), "acc": acc}
        if self._save_z:
            e["aux"] = {k: v.detach().to("cpu").clone() for k, v in state.aux.items()}
        self._latest[beta] = e
        tmp = f"{self._path}.tmp"
        torch.save({"levels": list(self._latest.values()), "betas": list(self._latest.keys()),
                    "config": self._config, "partial": True, "sweeps_done": self._sweeps}, tmp)
        os.replace(tmp, self._path)

    def init(self, n_particles, f, *, generator, device, dtype):
        st = super().init(n_particles, f, generator=generator, device=device, dtype=dtype)
        self._record(st, beta=0.0, lam=0.0, acc=float("nan"))
        return st

    def step(self, state, *, beta, lam, f, generator):
        acc = super().step(state, beta=beta, lam=lam, f=f, generator=generator)
        b = float(beta)
        self._sweeps += 1
        self._sweeps_at[b] = self._sweeps_at.get(b, 0) + 1
        self._record(state, beta=b, lam=float(lam), acc=float(acc))
        if time.time() > self._deadline:
            raise DeadlineReached(f"deadline after {self._sweeps} sweeps at beta={b:.4f}; "
                                  f"{len(self._latest)} levels on disk")
        return acc


class ResumePCNKernel(CheckpointingPCNKernel):
    """Starts the chain from a saved cloud instead of fresh draws from the latent prior.

    Correctness rests on the reward being the SAME function it was in the source run, so `init`
    recomputes f(X) and compares it against the checkpoint's stored fX. That costs one reward call
    (~1392 rows, ~6 min) and buys a direct verification that the references, gamma grid, Brownian
    bank and denoiser chunking all reproduced -- otherwise the run would sample, silently, from a
    slightly different target than the levels it is being stitched onto.

    THE TOLERANCE IS IN NATS OF LOG-WEIGHT, not in fractions of std_p(f). Job 697271 used
    `f_tol = 1e-4` of std, which at std_p(f) = 0.0156 means |df| < 1.6e-6 -- 1.5e-6 relative to
    f ~ 1.03. FLUX runs bf16, whose 8-bit mantissa carries ~0.2% per operation, so that bound was
    three orders of magnitude below what the arithmetic can deliver on ANY hardware. Stating it as a
    fraction of the std made it look reasonable only because this reward is unusually concentrated
    (std is 1.5% of the mean), which inflates round-off into a double-digit percentage.

    What actually matters is the SPREAD of the drift, not its size: q ∝ p·exp(lam·f) is invariant
    under f → f + c, so a uniform offset changes nothing, while a spread distorts relative weights by
    up to exp(lam' · spread). 0.05 nats is a 5% distortion of one particle's weight -- far below
    Monte-Carlo noise at N=16, and still tight enough to have caught 697271 (max|df| = 1.76e-3 would
    be up to 0.22 nats at lam' = 126.6).

    MEAN_TOL is a separate tripwire on the offset. It cannot corrupt the target, but a shift past
    bf16 round-off means a knob moved (wrong R, wrong gamma grid, wrong references), and that is
    worth aborting on even though the SMC would not notice.

    The guard is the backstop, not the defence. The defence is `--constraint=a6000` plus the
    persisted reference latents (cell 4), which together remove the two ways 697271 drifted.
    """

    MEAN_TOL = 0.01                  # of |mean f|; bf16 round-off ~0.2%, a wrong knob is well past 1%

    def __init__(self, *a, init_entry, lam, tol_nats=0.05, **kw):
        super().__init__(*a, **kw)
        self._init_entry, self._lam, self._tol_nats = init_entry, lam, tol_nats

    def init(self, n_particles, f, *, generator, device, dtype):
        e = self._init_entry
        X = e["X"].to(device=device, dtype=dtype)
        Z = e["aux"]["Z"].to(device=device, dtype=dtype)
        fX = f(X)                                        # recompute, do not trust the stored value
        fX_saved = e["fX"].to(device=device, dtype=dtype)
        d = fX - fX_saved
        f_mean = float(fX_saved.mean())
        offset = float(d.mean())                         # cancels in q; tripwire only
        spread = float(d.max() - d.min())                # this is what moves the relative weights
        nats = self._lam * spread
        print(f"reward reproducibility on {n_particles} particles:\n"
              f"  offset {offset:+.3e} ({offset / f_mean:+.3%} of f)   "
              f"max|df| {float(d.abs().max()):.3e} "
              f"({float(d.abs().max()) / float(fX_saved.std().clamp_min(1e-12)):.2%} of std_p(f))\n"
              f"  spread {spread:.3e} -> {nats:.4f} nats of log-weight at lam'={self._lam:.1f} "
              f"(tolerance {self._tol_nats})")
        if nats > self._tol_nats:
            raise RuntimeError(
                f"reward did not reproduce: {nats:.3f} nats of log-weight distortion > "
                f"{self._tol_nats} (spread {spread:.3e} at lam'={self._lam:.1f}). The resumed leg "
                f"would target a different distribution than the levels it is stitched onto. First "
                f"suspect is the GPU -- bf16 FLUX does not reproduce across architectures (this is "
                f"what killed 697271); then R / N_GAMMA / NUM_EPS / seeds / MAX_DENOISER_ROWS.")
        if abs(offset) > self.MEAN_TOL * abs(f_mean):
            raise RuntimeError(
                f"reward offset {offset / f_mean:+.2%} of f exceeds {self.MEAN_TOL:.0%}. A uniform "
                f"shift does not change q, but it is far past bf16 round-off (~0.2%), so a reward "
                f"knob has moved. Check R / N_GAMMA / NUM_EPS / gamma window / the reference set.")
        # logw = 0: the source cloud is post-resample and equally weighted (asserted in the knobs cell).
        st = _State(X=X, fX=fX, logw=torch.zeros(n_particles, device=device, dtype=dtype),
                    aux={"Z": Z})
        self._record(st, beta=0.0, lam=0.0, acc=float("nan"))
        return st


kernel = ResumePCNKernel(G, LATENT_DIM, init_entry=init_entry, lam=LAM_RESUME,
                         path=PARTIAL_CKPT, config=CONFIG, deadline=DEADLINE)
if "s" in init_entry:                       # only present if the source ran a post-fix notebook
    kernel.s = float(init_entry["s"])
    print(f"restored adapted pCN step size s = {kernel.s:.4f}")
else:
    print(f"source predates `s` persistence; starting from s0 = {kernel.s:.4f} and re-adapting "
          f"(695271 ended at acc = {init_entry['acc']:.2f} vs target {kernel.target_acc})")

print(f"\nSMC (resumed): N={N} R={R} N_gamma={N_GAMMA} beta0={BETA0:.4f} "
      f"lam'={LAM_RESUME:.1f} (of full {LAM_FULL:.1f})  ess_target={ESS_TARGET} "
      f"n_mcmc=None (adaptive, cap {MAX_SWEEPS})")
print(f"projected {t_sweep / 60:.1f} min/sweep; deadline "
      f"{time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")

t0, timed_out = time.time(), False
try:
    res = adaptive_tempering_smc_sample(
        reward, lam=LAM_RESUME, n_particles=N, kernel=kernel, ess_target=ESS_TARGET,
        n_mcmc=None, max_n_mcmc=MAX_SWEEPS, keep_levels=True, snapshot_device="cpu", seed=SEED,
    )
    levels = res.levels
    assert levels is not None
except DeadlineReached as exc:
    timed_out = True
    print(f"\n*** STOPPED ON DEADLINE: {exc}\n*** Rendering the levels that completed.")
    _e = list(kernel._latest.values())
    levels = [LevelSnapshot(beta=x["beta"], lam_eff=x["lam_eff"], X=x["X"], fX=x["fX"],
                            logw=x["logw"], ancestors=x["ancestors"],
                            ess=x.get("ess", float("nan"))) for x in _e]

    class _PartialResult:
        """Stands in for the result dataclass. ESS is read back from the kernel's per-level records
        (`state.ess`, set by the SMC loop before resampling). Stop reasons are INFERRED: all sweeps
        used -> 'cap', fewer -> 'decorr', and the level in progress when the deadline hit -> 'timeout'."""

        def __init__(s, entries):
            tail = entries[1:]
            s.betas = [x["beta"] for x in tail]
            s.ess_history = [x.get("ess", float("nan")) for x in tail]
            s.acc_history = [x["acc"] for x in tail]
            s.n_mcmc_history = [x["sweeps_at_beta"] for x in tail]
            s.stop_reason_history = [("cap" if x["sweeps_at_beta"] >= MAX_SWEEPS else "decorr")
                                     for x in tail]
            if s.stop_reason_history:
                s.stop_reason_history[-1] = "timeout"
            s.X, s.logw, s.levels = levels[-1].X, levels[-1].logw, levels

    res = _PartialResult(_e)

mins = (time.time() - t0) / 60
ckpt = os.path.join(SWEEP_DIR,
                    f"levels_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}"
                    f"{'_TIMEOUT' if timed_out else ''}.pt")
torch.save({
    "levels": [{"beta": s.beta, "lam_eff": s.lam_eff, "X": s.X, "fX": s.fX,
                "logw": s.logw, "ancestors": s.ancestors, "ess": s.ess} for s in levels],
    "X": res.X, "logw": res.logw, "betas": res.betas,
    "ess_history": res.ess_history, "acc_history": res.acc_history,
    "n_mcmc_history": res.n_mcmc_history, "stop_reason_history": res.stop_reason_history,
    "partial": timed_out,
    "config": {**CONFIG, "wall_clock_min": mins, "timed_out": timed_out},
}, ckpt)
print(f"\n{'TIMED OUT' if timed_out else 'completed'} in {mins:.1f} min ({mins / 60:.2f} h)   "
      f"levels={len(levels)} (this leg)")
print(f"saved -> {os.path.basename(ckpt)}  ({os.path.getsize(ckpt) / 2**20:.1f} MB)")
print(f"actual vs projected sweep cost: "
      f"{mins * 60 / max(1, sum(res.n_mcmc_history)):.1f} s vs {t_sweep:.1f} s")
print(f"beta reached (true): {BETA0 + levels[-1].beta * (1 - BETA0):.4f} of 1.0")


In [ ]:
# --- stitch: prior levels (from the source run) + this leg -----------------------------------------
# The SMC reported beta/lam_eff relative to the RESIDUAL leg, so both must be remapped before they mean
# anything:  beta_true = beta0 + u*(1-beta0)   and   lam_eff_true = beta0*LAM_FULL + lam_eff_run.
# Without this every row of this leg reads low by beta0*LAM_FULL = 150.0 (1.63 sigma).
#
# levels[0] of this leg IS src_levels[-1] (the same cloud), so it is dropped to avoid a duplicate row.
def _true_beta(u):
    return BETA0 + u * (1.0 - BETA0)


prior = [{"beta": e["beta"], "lam_eff": e["lam_eff"], "X": e["X"], "fX": e["fX"],
          "ess": e.get("ess", float("nan")), "acc": e.get("acc", float("nan")),
          "swp": e.get("sweeps_at_beta", 0), "leg": "695271"} for e in src_levels]
new = [{"beta": _true_beta(l.beta), "lam_eff": BETA0 * LAM_FULL + l.lam_eff, "X": l.X, "fX": l.fX,
        "ess": l.ess, "acc": float("nan"), "swp": 0, "leg": RUN_TAG} for l in levels[1:]]
for r, a, s in zip(new, res.acc_history, res.n_mcmc_history):
    r["acc"], r["swp"] = float(a), int(s)
allv = prior + new

lam_eff = [r["lam_eff"] for r in allv]
Eqf = [float(torch.as_tensor(r["fX"]).mean()) for r in allv]
f_sd = [float(torch.as_tensor(r["fX"]).std()) for r in allv]
uniq = [int(torch.as_tensor(r["X"]).unique(dim=0).shape[0]) / N for r in allv]
# LevelSnapshot.ess holds the RAW ESS count in [1, N] (post-reweight, pre-resample); normalize to
# compare against the 0.5 floor. Level 0 of each leg is a pre-reweight cloud, so nan.
ess = [r["ess"] / N for r in allv]
acc = [r["acc"] for r in allv]
base_f = Eqf[0]

print(f"this leg: betas(u) = {[round(b, 4) for b in res.betas]}")
print(f"          n_mcmc   = {res.n_mcmc_history}   (total {sum(res.n_mcmc_history)} sweeps)")
print(f"          stops    = {res.stop_reason_history}")
print(f"\nFULL LADDER ({len(prior)} levels from 695271 + {len(new)} from this leg)")
print(f"{'k':>2} {'leg':>10} {'beta':>7} {'lam_eff':>9} {'m':>6} {'E_q[f]':>8} {'d f':>8} "
      f"{'d f/sd':>7} {'ESS/N':>6} {'acc':>6} {'uniq/N':>7} {'f-sd':>8} {'swp':>4}")
for k, r in enumerate(allv):
    print(f"{k:2d} {r['leg']:>10} {r['beta']:7.4f} {lam_eff[k]:9.1f} {lam_eff[k] / lam_s:6.2f} "
          f"{Eqf[k]:8.4f} {Eqf[k] - base_f:+8.4f} {(Eqf[k] - base_f) / f_sd[0]:7.1f} "
          f"{ess[k]:6.2f} {acc[k]:6.2f} {uniq[k]:7.2f} {f_sd[k]:8.4f} {r['swp']:4d}")

print(f"\nnovelty gain over the FULL ladder: E_q[f] {base_f:.4f} -> {Eqf[-1]:.4f} "
      f"({100 * (Eqf[-1] / base_f - 1):+.1f}%, {(Eqf[-1] - base_f) / f_sd[0]:.1f} sigma_p(f))")
print(f"beta {allv[-1]['beta']:.4f} of 1.0   m_eff {lam_eff[-1] / lam_s:.2f} of {M_TILT:.0f}")
n_cap = sum(1 for r in res.stop_reason_history if r == "cap")
print(f"this leg hit the sweep cap ({MAX_SWEEPS}): {n_cap}/{len(res.stop_reason_history)}"
      f"   (695271: 0/4 at cap 24 -- it was never binding there)")
print("\nNOTE: ESS/N sits at ESS_TARGET by construction -- _next_dbeta bisects d_beta to place it")
print("there exactly. It is a schedule sanity check, not a degeneracy signal; the exceptions are the")
print("FINAL level (d_beta = 1-beta, so ESS floats above target) and any level where the bisection")
print("bottoms out. Read degeneracy off acc, uniq/N and f-spread instead.")

# --- diagnostics ----------------------------------------------------------------------------------
split = len(prior) - 0.5      # boundary between the two legs, drawn on every panel
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, (y, lbl, ttl, col) in zip(ax, [
        (Eqf, r"$E_q[f]$", "novelty vs tilt", "tab:blue"),
        (None, None, "diversity", None),
        (acc, "acceptance", "rejuvenation health", "darkorange"),
        (f_sd, r"std$_q(f)$", "f-spread (collapse uniq/N misses)", "darkgreen")]):
    if y is None:
        a.plot(lam_eff, uniq, "o-", label="uniq/N")
        a.plot(lam_eff, ess, "s--", label="ESS/N")
        a.axhline(0.5, color="crimson", ls=":", label="0.5 floor")
        a.axhline(ESS_TARGET, color="grey", ls="-.", lw=1, label=f"ess_target={ESS_TARGET}")
        a.set_ylim(0, 1.05); a.legend(fontsize=7)
    else:
        a.plot(lam_eff, y, "o-", color=col); a.set_ylabel(lbl)
    a.axvline(BETA0 * LAM_FULL, color="k", ls="--", lw=1, alpha=0.5)
    a.set_title(ttl); a.set_xlabel(r"$\lambda_{eff}$"); a.grid(alpha=0.3)
ax[2].axhline(0.23, color="grey", ls=":", label="pCN target"); ax[2].legend(fontsize=7)
fig.suptitle(f"full ladder: 695271 (left of dashed) + {RUN_TAG} (right)", fontsize=11)
plt.tight_layout(); plt.show()

fig, a = plt.subplots(figsize=(8.5, 3.4))
cols = ["tab:red" if r == "cap" else "tab:grey" if r == "timeout" else "tab:blue"
        for r in res.stop_reason_history]
a.bar(range(len(res.betas)), res.n_mcmc_history, color=cols, alpha=0.85)
a.axhline(MAX_SWEEPS, ls="--", color="tab:red", lw=1, label=f"cap = {MAX_SWEEPS}")
a.set_xticks(range(len(res.betas)))
a.set_xticklabels([f"{_true_beta(b):.3f}" for b in res.betas], rotation=45, fontsize=8)
a.set_xlabel(r"$\beta$ (true)"); a.set_ylabel("sweeps"); a.legend(fontsize=8); a.grid(alpha=0.3, axis="y")
a.set_title(f"rejuvenation effort, resumed leg only  (red = hit cap, grey = timeout)")
plt.tight_layout(); plt.show()


In [ ]:
# --- images: the FULL ladder, both legs in one grid ------------------------------------------------
# Column order is raw particle index, not lineage. The source run's checkpointed `ancestors` are arange
# (resampling happens outside the kernel), so lineage cannot be tracked across the join -- claiming a
# column follows one particle through both legs would be false. Rows remain comparable; columns are not.
decoded = [decode(torch.as_tensor(r["X"])) for r in allv]
print(f"decoded {len(allv)} levels x {N} particles "
      f"({len(prior)} from 695271, {len(new)} from this leg)")

nrow = len(allv)
fig, axes = plt.subplots(nrow, N, figsize=(N * 1.5, nrow * 1.65), squeeze=False)
for k in range(nrow):
    for j in range(N):
        a = axes[k][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        if j == 0:
            a.text(-0.14, 0.5,
                   f"$\\lambda_{{eff}}$={lam_eff[k]:.0f}\n$\\beta$={allv[k]['beta']:.3f}\n"
                   f"$E_q[f]$={Eqf[k]:.3f}",
                   transform=a.transAxes, ha="right", va="center", fontsize=7,
                   color=("black" if allv[k]["leg"] == "695271" else "tab:blue"))
        if k == 0:
            a.set_title(f"col {j}", fontsize=7)
fig.suptitle(f"FULL LADDER  m={M_TILT:.0f}, $\\lambda_{{max}}$={LAM_FULL:.0f}   prompt {PROMPT!r}, N={N}\n"
             f"rows: $\\lambda_{{eff}}$ ascending (top = untilted $p$)   "
             f"black labels = job 695271, blue = {RUN_TAG}", fontsize=12)
plt.tight_layout(rect=(0.05, 0, 1, 0.965))
plt.show()
fig.savefig(GRID_PNG, dpi=110, bbox_inches="tight")

# Untilted anchor vs most tilted, enlarged, for a direct visual read on what the tilt did.
for k, label in ((0, "UNTILTED $\\lambda_{eff}=0$ (samples from $p$)"),
                 (len(allv) - 1, f"MOST TILTED $\\lambda_{{eff}}$={lam_eff[-1]:.0f} "
                                 f"(m={lam_eff[-1] / lam_s:.1f}, $\\beta$={allv[-1]['beta']:.3f})")):
    fig, axes = plt.subplots(1, N, figsize=(N * 1.7, 2.4), squeeze=False)
    for j in range(N):
        a = axes[0][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        a.set_title(f"f={float(torch.as_tensor(allv[k]['fX'])[j]):.4f}", fontsize=7)
    fig.suptitle(label, fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.90)); plt.show()

# Per-image PNGs for full-resolution inspection, into a tagged directory.
from PIL import Image

os.makedirs(DECODED_DIR, exist_ok=True)
for k in range(len(allv)):
    for j in range(N):
        arr = (decoded[k][j].permute(1, 2, 0).clamp(0, 1) * 255).round().to(torch.uint8).numpy()
        Image.fromarray(arr).save(
            os.path.join(DECODED_DIR, f"lvl{k:02d}_lam{lam_eff[k]:.0f}_p{j}.png"))
print(f"wrote {len(allv) * N} PNGs to {os.path.basename(DECODED_DIR)}/ "
      f"and {os.path.basename(GRID_PNG)}")


## Takeaways

*(Fill in after the run — this cell is the authoritative statement of what was learned, per repo convention.)*

**The question.** 695271 established that the style axis survives a converged $f$ ($N_\gamma = 30$), but stopped
at $\beta = 0.5422$, $m_{\text{eff}} = 1.63$ of 3. This leg asks what the top of the ladder looks like — and
whether the sampler can even get there, given acceptance fell 0.75 → 0.62 → 0.38 → 0.12 over the four levels.

**To record:**

- $\beta$ and $m_{\text{eff}}$ reached; whether $\beta = 1$ was attained or the deadline truncated again
- **Reward reproducibility check** — the `max|f_new - f_saved|` printed at init. Non-trivial drift would mean
  the resumed leg targets a different distribution than the levels it is stitched onto, and invalidates the
  combined ladder
- **Did acceptance keep falling?** 0.12 at $\lambda_{\text{eff}} = 150$ was already below the pCN target of
  0.23. If it collapses further, the top levels are not draws from $q_\lambda$ and should be labelled as such
- **Did `MAX_SWEEPS = 40` start binding?** It never did at 24 in 695271. Levels stopping on `cap` rather than
  `decorr` is the signal that rejuvenation is now the constraint rather than wall-clock
- $\Delta\beta$ per level vs 695271's 0.136: does the schedule keep advancing, or does bisection start
  returning ever-smaller steps as it did in 694278 (0.019 and shrinking)?
- Actual vs projected sweep cost against `S_ROW = 0.251` — a third validation of the cost model
- Visual verdict on the **full** ladder: does the style trend continue past $\lambda_{\text{eff}} = 150$, or
  does it saturate / degenerate?

**On ESS.** Now recorded rather than NaN, but it sits at `ESS_TARGET` by construction — `_next_dbeta` bisects
$\Delta\beta$ to place it there exactly. Treat it as a check that the schedule is behaving, not as a
degeneracy signal, and watch the final level (where $\Delta\beta = 1-\beta$, so ESS floats above target).
Degeneracy still shows up first in `acc` and `uniq/N`.

**Caveats.** The combined ladder's rows come from two jobs; **columns are not lineage-tracked across the
join** (the source checkpoint's `ancestors` are `arange`), so a column is not one particle followed through
both legs. Rows still share ancestry within each leg and are not independent draws from
$q_{\lambda_{\text{eff}}}$. σ units are not comparable to runs with a different $\gamma$ grid. The pCN step
size restarted from $s_0$ if the source predates `s` persistence — the first few sweeps of this leg pay a
re-adaptation cost that the acceptance history will show.
